In [ ]:
"""Normal-order X^N with X = phi_a (a + a†) + phi_b (b + b†).

Output: dict {(m, n, p, q): coeff} for the monomial a†^m a^n b†^p b^q.
In the rotating frame each monomial carries e^{ik theta} with k = (m-n) + 2(p-q).
"""
import sympy as sp
from math import comb, factorial
from collections import defaultdict

phi_a, phi_b = sp.symbols("phi_a phi_b", positive=True)


def _mul_mode(m1, n1, m2, n2):
    """a†^m1 a^n1 · a†^m2 a^n2 = sum_r r! C(n1,r) C(m2,r) a†^(m1+m2-r) a^(n1+n2-r)."""
    return {(m1 + m2 - r, n1 + n2 - r): factorial(r) * comb(n1, r) * comb(m2, r)
            for r in range(min(n1, m2) + 1)}


def mul(A, B):
    """Product of two normal-ordered operators (dicts); the a and b parts factorize."""
    C = defaultdict(int)
    for (m1, n1, p1, q1), c1 in A.items():
        for (m2, n2, p2, q2), c2 in B.items():
            for (m, n), ca in _mul_mode(m1, n1, m2, n2).items():
                for (p, q), cb in _mul_mode(p1, q1, p2, q2).items():
                    C[(m, n, p, q)] += c1 * c2 * ca * cb
    return {k: sp.expand(v) for k, v in C.items() if sp.expand(v) != 0}


def X_power(N):
    """Normal-ordered X^N as {(m, n, p, q): coeff}."""
    X = {(1, 0, 0, 0): phi_a, (0, 1, 0, 0): phi_a,
         (0, 0, 1, 0): phi_b, (0, 0, 0, 1): phi_b}
    out = {(0, 0, 0, 0): sp.Integer(1)}
    for _ in range(N):
        out = mul(out, X)
    return out


def add(A, B, s=1):
    """A + s*B for operator dicts."""
    C = defaultdict(int)
    for k, v in A.items():
        C[k] += v
    for k, v in B.items():
        C[k] += s * v
    return {k: sp.expand(v) for k, v in C.items() if sp.expand(v) != 0}


def scale(A, c):
    return {k: sp.expand(c * v) for k, v in A.items()}


def comm(A, B):
    """[A, B] for operator dicts."""
    return add(mul(A, B), mul(B, A), -1)


def harmonics(V):
    """Split {(m,n,p,q): c} into {k: {(m,n,p,q): c}} with k = (m-n) + 2(p-q)."""
    Hk = defaultdict(dict)
    for (m, n, p, q), c in V.items():
        Hk[(m - n) + 2 * (p - q)][(m, n, p, q)] = c
    return dict(Hk)


def merge_harmonics(*Hks):
    """Merge several {k: dict} (e.g. sine part + drive terms with explicit k)."""
    out = defaultdict(dict)
    for Hk in Hks:
        for k, A in Hk.items():
            out[k] = add(out[k], A)
    return dict(out)


def second_order(Hk, Omega):
    """H^(2) = 1/2 sum_{k != 0} [H_k, H_{-k}] / (k Omega), convention H(t) = sum_k H_k e^{ik theta}."""
    H2 = {}
    for k, A in Hk.items():
        if k == 0 or -k not in Hk:
            continue
        H2 = add(H2, scale(comm(A, Hk[-k]), sp.Rational(1, 2) / (k * Omega)))
    return H2

In [ ]:
g, g_0, Omega, eps_d = sp.symbols("g g_0 Omega epsilon_d", positive=True)

X3 = X_power(3)
print("Normal-ordered X^3:")
for key in sorted(X3):
    m, n, p, q = key
    print(" ", key, "k =", (m - n) + 2 * (p - q), " coeff =", sp.factor(X3[key]))

In [ ]:
# Order-phi^3 nonlinearity V = (g_0/6) X^3, split into harmonics
V = scale(X3, g_0 / 6)
Hk = harmonics(V)

# Counter-rotating drive: explicit k that differs from the shift-derived one
drive = {-4: {(0, 0, 0, 1): eps_d / 2}, 4: {(0, 0, 1, 0): eps_d / 2}}
Hk_full = merge_harmonics(Hk, drive)

H2 = second_order(Hk_full, Omega)
print("\nH^(2):")
for key in sorted(H2):
    print(" ", key, " coeff =", sp.factor(H2[key]).subs(g_0, 2*g/(phi_a**2*phi_b)).simplify())

In [ ]:
import json
import pathlib

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def _find_repo():
    """Locate the repo checkout without hardcoding an absolute path.

    Notebooks have no __file__, so search upward from the working directory --
    which VS Code sets to either the notebook's folder or the workspace root.
    """
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "src" / "floquet_lindblad").is_dir():
            return base
        for hit in sorted(base.glob("*/src/floquet_lindblad")):
            return hit.parents[1]
    raise RuntimeError(f"no checkout with src/floquet_lindblad found above {here}")


REPO = _find_repo()

# job name -> (file, factor from the file's `alpha_sq` to |alpha|^2).
# results_small.json predates the 2026-09 alpha_sq fix (docs/models_ats.md,
# section 2): its labels are twice the cat size, so 0.5 puts both files on one
# grid. That grid is nominal only -- see the truncation section below.
# RWA (RWA/notebooks/sweep_alpha_eps_p_RWA_0.ipynb) is not an ATS truncation but
# the separate time-independent RWA model, run on `ref`'s (alpha_sq, eps_p) grid
# with the same `params` (plus alpha_sq = 7), so it compares with `ref` point by
# point -- see the RWA section below.
FILES = {
    "RWA": (REPO / "results" / "sweep_alpha_eps_p_RWA_0_ref.json", 1.0),
}


def _complex_hook(d):
    """{"real", "imag"} -> complex, as floquet_lindblad.io.from_jsonable does.

    That module imports jax, which this client does not have.
    """
    return complex(d["real"], d["imag"]) if d.keys() == {"real", "imag"} else d


def load_sweep(path, to_cat_size):
    with open(path) as f:
        pts = json.load(f, object_hook=_complex_hook)
    for r in pts:
        r["alpha_sq_file"] = r["alpha_sq"]
        r["alpha_sq"] = r["alpha_sq"] * to_cat_size
        # linspace leaves 0.30000000000000004 and the like; round so eps_p works as a key.
        r["eps_p"] = round(r["eps_p"], 12)
        if "params" in r:
            # The true cat size at leading order, whichever convention wrote the file.
            r["cat_size"] = r["params"]["epsilon_d"] / (2 * r["params"]["g"])
        for key in ("res_hist", "apply_hist", "mu_hist", "fock_weight_a", "fock_weight_b"):
            if key in r:
                r[key] = np.asarray(r[key])
    return pts


results = {name: load_sweep(path, f) for name, (path, f) in FILES.items()}

ALPHA_SQ_LIST = sorted({r["alpha_sq"] for res in results.values() for r in res})
EPS_P_LIST = sorted({r["eps_p"] for res in results.values() for r in res})
EPS_D_LIST = sorted({r["params"]["epsilon_d"] for res in results.values() for r in res})
g_LIST = sorted({r["params"]["g"] for res in results.values() for r in res})

In [ ]:
import ast


def _ats_constants(*names):
    """Read module constants from models/ats.py without importing it.

    Importing floquet_lindblad pulls in jax and dynamiqs, which this client does
    not have; parsing the source keeps ats.py the single source of truth.
    """
    tree = ast.parse((REPO / "src" / "floquet_lindblad" / "models" / "ats.py").read_text())
    found = {}
    for node in tree.body:
        if isinstance(node, ast.Assign) and len(node.targets) == 1:
            target = node.targets[0]
            if isinstance(target, ast.Name) and target.id in names:
                found[target.id] = eval(compile(ast.Expression(node.value), "ats.py", "eval"), {"np": np})
    missing = set(names) - found.keys()
    if missing:
        raise KeyError(f"not found in ats.py: {sorted(missing)}")
    return tuple(found[n] for n in names)


W_A, PHI_A, PHI_B = _ats_constants("W_A", "PHI_A", "PHI_B")

In [ ]:
# Second-order coefficients from H^(2) above, with Omega = W_A.
# K_a, K_b, chi, D_a, D_b depend on g only, i.e. on eps_p: one value per eps_p.
pts = results["RWA"]
g_of_eps_p = {r["eps_p"]: r["params"]["g"] for r in pts}
g_arr = np.array([g_of_eps_p[e] for e in EPS_P_LIST])

Ka  = g_arr**2 / W_A * (40 * PHI_A**2 + 27 * PHI_B**2) / (6 * PHI_B**2)
Kb  = g_arr**2 / W_A * 2 * PHI_B**2 * (58 * PHI_A**2 + 25 * PHI_B**2) / (15 * PHI_A**4)
chi = g_arr**2 / W_A * (135 * PHI_A**2 + 92 * PHI_B**2) / (15 * PHI_A**2)
Da  = -g_arr**2 / W_A * (100 * PHI_A**4 + 135 * PHI_A**2 * PHI_B**2 + 46 * PHI_B**4) / (15 * PHI_A**2 * PHI_B**2)
Db  = -g_arr**2 / W_A * (135 * PHI_A**4 + 324 * PHI_A**2 * PHI_B**2 + 100 * PHI_B**4) / (30 * PHI_A**4)

# eps_2 = -eps_d g / (8 W_A) depends on both eps_d and g: grid over (alpha_sq, eps_p).
eps2 = np.full((len(ALPHA_SQ_LIST), len(EPS_P_LIST)), np.nan)
for r in pts:
    i, j = ALPHA_SQ_LIST.index(r["alpha_sq"]), EPS_P_LIST.index(r["eps_p"])
    eps2[i, j] = -r["params"]["epsilon_d"] * r["params"]["g"] / (8 * W_A)

In [ ]:
fig, ax = plt.subplots()
for y, label in [(Ka/W_A, r"$K_a/\omega_a$"), (Kb/W_A, r"$K_b/\omega_a$"), (chi/W_A, r"$\chi/\omega_a$"), (Da/W_A, r"$\Delta_a/\omega_a$"), (Db/W_A, r"$\Delta_b/\omega_a$")]:
    ax.plot(EPS_P_LIST, y, "o-", label=label)
ax.set_xlabel(r"$\epsilon_p$")
ax.set_ylabel("Coefficient")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots()
mesh = ax.pcolormesh(EPS_P_LIST, ALPHA_SQ_LIST, eps2/W_A, shading="nearest")
fig.colorbar(mesh, ax=ax, label=r"$\epsilon_2/\omega_a$")
ax.set_xlabel(r"$\epsilon_p$")
ax.set_ylabel(r"$|\alpha|^2$")
plt.show()